In [16]:
import pandas as pd
import random 
from faker import Faker
import uuid

In [17]:
fake = Faker()

In [18]:
products = pd.read_csv(r"C:\Users\DELL\Desktop\product.csv", encoding = "latin1")
warehouses = pd.read_csv(r"C:\Users\DELL\Desktop\warehouse.csv", encoding = "latin1")

In [19]:
fast_categories = [
    "Grocery",
    "Beverages",
    "Fitness"
]

medium_categories = [
    "Electronics",
    "Fashion",
    "Stationery"
]

slow_categories = [
    "Sports",
    "Toys"
]

In [20]:
warehouse_distribution = {
    "Grocery": 20,
    "Beverages": 17,
    "Fitness": 15,
    "Stationery": 12,
    "Fashion": 10,
    "Electronics": 8,
    "Sports": 10,
    "Toys": 4
}

In [21]:
def generate_available_qty(category):

    if category in fast_categories:
        return random.randint(300,700)

    elif category in medium_categories:
        return random.randint(120,350)

    else:
        return random.randint(20,120)

In [22]:
def generate_reorder_qty(category):

    if category in fast_categories:
        return random.randint(100, 250)

    elif category in medium_categories:
        return random.randint(40, 120)

    else:
        return random.randint(10, 50)

In [23]:
inventory = []

warehouse_ids = warehouses["id"].tolist()

for _, product in products.iterrows():

    category = product["category"]

    num_warehouses = warehouse_distribution[category]

    # Select unique warehouses
    selected_warehouses = random.sample(
        warehouse_ids,
        num_warehouses
    )

    for warehouse_id in selected_warehouses:

        # Available Quantity
        available_qty = generate_available_qty(category)

        # Reserved Quantity (0-30%)
        reserved_qty = round(
            available_qty * random.randint(0, 30) / 100
        )

        # Initial Reorder Quantity
        reorder_qty = generate_reorder_qty(category)

        # 22% records should have reorder pending
        if random.random() < 0.22:

            # Force availableQty <= reorderQty
            if available_qty > reorder_qty:
                reorder_qty = available_qty + random.randint(1, 30)

        else:

            # Force availableQty > reorderQty
            if available_qty <= reorder_qty:
                reorder_qty = max(
                    1,
                    available_qty - random.randint(1, 30)
                )

        # Boolean Column
        is_reorder_pending = available_qty <= reorder_qty

        inventory.append({

            "id": str(uuid.uuid4()),

            "productId": product["id"],

            "warehouseId": warehouse_id,

            "availableQty": available_qty,

            "reservedQty": reserved_qty,

            "reorderQty": reorder_qty,

            "isReorderPending": is_reorder_pending

        })

In [24]:
inventory_df = pd.DataFrame(inventory)

In [25]:
inventory_df.to_csv(
    "inventory.csv",
    index = False
)